# Adapting AlphaGenome to RNA-seq data with backbone LoRA

Heads-only training learns a fresh head on frozen backbone embeddings. Backbone
**LoRA (Low-Rank Adaptation)** also adapts the query and value projections in all
nine transformer blocks while keeping pretrained backbone weights frozen.

This example trains those adapters together with a normal RNA prediction head.
The custom head uses softplus predictions and MSE on raw BigWig targets; use a
predefined RNA head for AlphaGenome's native track scaling and loss.

See [LoRA adapters](../docs/lora_adapters.md) for training defaults and
checkpoint compatibility.

## 1. Install and import dependencies

For a local checkout, skip installation and use the environment containing that checkout.

In [ ]:
# Installation will take a few minutes
%pip install -q "jax[cuda12]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
%pip install -q "alphagenome>=0.7.0,<2"
%pip install -q git+https://github.com/google-deepmind/alphagenome_research.git
%pip install -q git+https://github.com/genomicsxai/alphagenome_ft.git
print("Done.")

In [ ]:
from __future__ import annotations

from pathlib import Path
import gzip
import shutil
import urllib.request

import jax
import jax.numpy as jnp
import haiku as hk
from alphagenome.models import dna_output

from alphagenome_ft import (
    BackboneLoRAConfig,
    create_model_with_heads,
    register_custom_head,
    CustomHead,
    CustomHeadConfig,
    CustomHeadType,
    lora,
    parameter_utils,
)
from alphagenome_ft.backbone_lora import expected_lora_parameter_count
from alphagenome_ft.finetune.config import load_targets_config, prepare_head_specs
from alphagenome_ft.finetune.data import (
    prepare_intervals_from_fold,
    BigWigDataModule,
    build_fasta_index,
)
from alphagenome_ft.finetune.train import train

## 2. Define the user config

Edit the following config values to specify data paths and training hyperparameters.

In [ ]:
# Core input files
PROJECT_DIR = Path("/path/to/project")
FASTA_PATH = Path("/path/to/genome.fa")

# Head identifier – used in checkpoints and as the BigWig target key
HEAD_NAME = "lora_rna_head"

# Target BigWig files (must be listed in the same order as NUM_TRACKS)
TARGET_BIGWIGS = [
    PROJECT_DIR / "example_rna_rep1_chr1.bw",
    PROJECT_DIR / "example_rna_rep2_chr1.bw"
]
NUM_TRACKS = len(TARGET_BIGWIGS)  # must match number of BigWig files above

# LoRA hyperparameters
# rank: dimension of the low-rank matrices A and B
# alpha: scaling factor; effective scale applied to the LoRA delta is alpha / rank
#        setting alpha == rank gives a scale of 1.0
LORA_RANK  = 8
LORA_ALPHA = 16.0
LORA_TARGETS = ("q_proj", "v_proj")
GRADIENT_CHECKPOINTING = True

# Data options
FOLD        = "1"         # 0, 1, 2, 3
WINDOW_SIZE = 1_048_576   # 1 Mbp window

# Model options
ORGANISM      = "HOMO_SAPIENS"
MODEL_VERSION = "fold_1"  # fold_0, fold_1, fold_2, fold_3, all_folds

# Training options
BATCH_SIZE       = 1
NUM_EPOCHS       = 10
LEARNING_RATE    = 3e-4
WEIGHT_DECAY     = 0.0
MAX_GRAD_NORM    = 1.0
SEED             = 42
SHUFFLE          = True
DROP_LAST        = False
MAX_TRAIN_STEPS  = None
VERBOSE          = True

# Checkpointing / early stopping
CHECKPOINT_DIR          = PROJECT_DIR / "checkpoints/lora_demo"
BEST_METRIC             = "valid_loss"
BEST_METRIC_MODE        = "min"
EARLY_STOPPING_PATIENCE = 5
EARLY_STOPPING_MIN_DELTA = 0.0

## 3. Optional data download

Download genome FASTA file and the RNA bigwig files from `alphagenome_ft/data` on GitHub when missing.

In [ ]:
ENABLE_DATA_DOWNLOAD  = True
ENABLE_FASTA_DOWNLOAD = True

GITHUB_DATA_BASE = "https://raw.githubusercontent.com/genomicsxai/alphagenome_ft/main/data"
DEMO_FILES = ["example_rna_rep1_chr1.bw", "example_rna_rep2_chr1.bw"]
FASTA_DOWNLOAD_URL = "https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz"

PROJECT_DIR.mkdir(parents=True, exist_ok=True)

if ENABLE_DATA_DOWNLOAD:
    for filename in DEMO_FILES:
        dst = PROJECT_DIR / filename
        if dst.exists():
            print(f"Skip existing: {dst}")
            continue
        src = f"{GITHUB_DATA_BASE}/{filename}"
        print(f"Downloading: {src} -> {dst}")
        urllib.request.urlretrieve(src, dst)
else:
    print("Core data download is disabled.")

if ENABLE_FASTA_DOWNLOAD:
    if FASTA_PATH.exists():
        print(f"Skip existing: {FASTA_PATH}")
    else:
        gz_path = FASTA_PATH.with_suffix(FASTA_PATH.suffix + ".gz")
        print(f"Downloading: {FASTA_DOWNLOAD_URL} -> {gz_path}")
        urllib.request.urlretrieve(FASTA_DOWNLOAD_URL, gz_path)
        print(f"Decompressing: {gz_path} -> {FASTA_PATH}")
        with gzip.open(gz_path, "rb") as src, FASTA_PATH.open("wb") as dst:
            shutil.copyfileobj(src, dst)
        gz_path.unlink()
else:
    print("FASTA download is disabled.")

FASTA_INDEX_PATH = Path(f"{FASTA_PATH}.fai")
if FASTA_PATH.exists() and not FASTA_INDEX_PATH.exists():
    print(f"Building FASTA index: {FASTA_INDEX_PATH}")
    build_fasta_index(FASTA_PATH)
elif FASTA_INDEX_PATH.exists():
    print(f"Found FASTA index: {FASTA_INDEX_PATH}")
else:
    print("Skip FASTA indexing because FASTA is missing.")

## 4. Validate inputs

In [ ]:
for path in [FASTA_PATH, FASTA_INDEX_PATH, *TARGET_BIGWIGS]:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

if CHECKPOINT_DIR is not None:
    CHECKPOINT_DIR = Path(CHECKPOINT_DIR).resolve()
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("Input validation passed.")

## 5. Define and register the RNA prediction head

The head projects 1 bp embeddings to `NUM_TRACKS` channels with a normal
`hk.Linear` layer, then applies softplus. Its weights and bias are trainable.
LoRA lives in the backbone attention projections, configured in step 7;
zero-initialized adapter B matrices preserve the pretrained trunk at initialization.

In [ ]:
class RNAHead(CustomHead):
    """Trainable linear RNA head on the LoRA-adapted backbone embeddings."""

    def predict(self, embeddings, organism_index, **kwargs):
        x = embeddings.get_sequence_embeddings(resolution=1)
        logits = hk.Linear(self._num_tracks, name="output")(x)
        return {"predictions": jax.nn.softplus(logits)}

    def loss(self, predictions, batch):
        preds = predictions["predictions"]
        targets = batch["targets"]
        return {"loss": jnp.mean(jnp.square(preds - targets))}

register_custom_head(
    HEAD_NAME,
    RNAHead,
    head_config=CustomHeadConfig(
        type=CustomHeadType.GENOME_TRACKS,
        output_type=dna_output.OutputType.RNA_SEQ,
        num_tracks=NUM_TRACKS,
    ),
)
print(f"Registered custom head: {HEAD_NAME!r}")

## 6. Build head specs and fold intervals

Specify the targets config. `source: custom` tells `prepare_head_specs` to look up
the head in the registry populated by `register_custom_head` above.

In [ ]:
TARGETS_CONFIG = {
    "heads": [
        {
            "id": HEAD_NAME,
            "source": "custom",
            "targets": [{"path": str(bw)} for bw in TARGET_BIGWIGS],
        }
    ]
}

config_dict = load_targets_config(TARGETS_CONFIG)
head_specs  = prepare_head_specs(config_dict)

intervals = prepare_intervals_from_fold(
    fold=FOLD,
    window_size=WINDOW_SIZE,
    organism=ORGANISM,
)

print("Heads:", [spec.head_id for spec in head_specs])
for split in ("train", "valid", "test"):
    print(f"{split}: {len(intervals.get(split, []))} intervals")

## 7. Initialize the model

### Kaggle credentials (required for downloading AlphaGenome weights)

The next cell downloads the base Alphagenome model weights from Kaggle when creating the backbone-LoRA model. 

**One-time setup:**

1. **Accept the dataset agreement:** [AlphaGenome on Kaggle](https://www.kaggle.com/models/google/alphagenome) → open the page and accept terms if prompted.
2. **Create a _legacy_ API token:** [Kaggle Account → Create New Token](https://www.kaggle.com/settings) (downloads `kaggle.json`). Use the **username** and **key** from that file (the **key** is the long token, not your account password). 

**NOTE** - do not use a 'API Tokens (Recommended)' as this will **not** work.

Once you log in and it's successful, rerun the code to create the model.

In [ ]:
lora_config = BackboneLoRAConfig(
    rank=LORA_RANK,
    alpha=LORA_ALPHA,
    targets=LORA_TARGETS,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
)
model = create_model_with_heads(
    MODEL_VERSION,
    heads=[HEAD_NAME],
    backbone_lora_config=lora_config,
    init_seq_len=2**14,
)

print("Model ready: backbone Q/V adapters plus a trainable RNA head.")

## 8. Inspect the backbone and LoRA parameter paths

Inspect the pretrained backbone and its new LoRA adapters.  `lora.get_lora_parameter_paths` finds every leaf named `lora_a` or `lora_b`;
`lora.count_lora_parameters` sums their element counts so you can verify the adapter
is sufficiently lightweight for fine-tuning.

In [ ]:
backbone_paths = parameter_utils.get_backbone_parameter_paths(model._params)
print(f"Backbone parameter arrays : {len(backbone_paths):,}")
print("Sample backbone paths:")
for p in backbone_paths[:6]:
    print("  ", p)
print()

In [ ]:
lora_paths = lora.get_lora_parameter_paths(model._params)
print(f"LoRA adapter arrays: {len(lora_paths)}")
for p in lora_paths:
    print("  ", p)
print()

total_params = parameter_utils.count_parameters(model._params)
lora_params  = lora.count_lora_parameters(model._params)
print(f"Total parameters : {total_params:>14,}")
print(f"LoRA parameters  : {lora_params:>14,}")
print(f"LoRA fraction    : {lora_params / total_params:.4%}")

assert lora_params == expected_lora_parameter_count(rank=LORA_RANK, targets=LORA_TARGETS)
assert all("transformer_tower" in path for path in lora_paths)
print("Verified adapters are in the transformer backbone (308,736 scalars at rank 8, Q/V).")

## 9. Inspect the trainable head

The trainer detects `model._backbone_lora_config` and defaults to
`heads_only=True, train_lora=True`. It stops frozen backbone gradients inside the
differentiated function and masks optimizer updates, while training all head and
adapter leaves. No manual `freeze_except_lora` call is needed; applying
`stop_gradient` to arrays before a later `jax.grad` does not persistently freeze them.

In [ ]:
head_paths = parameter_utils.get_head_parameter_paths(model._params)
print(f"Trainable head parameter arrays: {len(head_paths)}")
for p in head_paths:
    print("  ", p)

## 10. Build the BigWig data module

In [ ]:
data_module = BigWigDataModule(
    intervals=intervals,
    fasta_path=FASTA_PATH,
    head_specs=head_specs,
    batch_size=BATCH_SIZE,
    shuffle=SHUFFLE,
    drop_last=DROP_LAST,
)

print("Data module ready.")

## 11. (Optional) Smoke test settings

Use this to verify the full pipeline quickly before a long run.

In [ ]:
SMOKE_TEST = True
if SMOKE_TEST:
    NUM_EPOCHS      = 1
    MAX_TRAIN_STEPS = 20
    print("Smoke test enabled.")
else:
    print("Smoke test disabled.")

## 12. Train the model

With this backbone-LoRA model, leaving `heads_only` and `train_lora` unspecified
trains the RNA head and all adapters, keeping the pretrained backbone frozen.
Gradient clipping uses only trainable leaves. Set `MAX_GRAD_NORM` to `None` or
`0` to disable clipping. `train_lora=True, heads_only=False` is rejected.

Default delta checkpoints contain the head and adapters; reload them with the
same fold-specific base model and registered head. Full-model checkpoints also
retain LoRA. Minimal encoder-only checkpoints are incompatible with backbone LoRA.

In [ ]:
train(
    model,
    data_module,
    head_specs,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    num_epochs=NUM_EPOCHS,
    seed=SEED,
    max_train_steps=MAX_TRAIN_STEPS,
    max_grad_norm=MAX_GRAD_NORM,
    checkpoint_dir=CHECKPOINT_DIR,
    organism=ORGANISM,
    best_metric=BEST_METRIC,
    best_metric_mode=BEST_METRIC_MODE,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
    verbose=VERBOSE,
)